# WavLM + Prosody Training - EXACT Kaggle Parameters
## 555 videos, same architecture as working 71-video model

**Key parameters (from Kaggle notebook):**
- MAX_DURATION: 5.0s
- BATCH: 16
- EPOCHS: 10
- WavLM: FROZEN + AttentionPooling
- Prosody→32d: Linear(21,64) → GELU → Dropout(0.1) → Linear(64,32)
- Classifier: Dropout(0.3) → Linear(800,128) → GELU → Dropout(0.2) → Linear(128,64) → GELU → Linear(64,2)
- Optimizer: AdamW(lr=1e-3, weight_decay=0.01)
- Loss: CrossEntropyLoss(weight=[1.0, 2.5])
- Gradient clip: max_norm=1.0

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted')

In [ ]:
!pip install -q transformers datasets torch torchaudio librosa scikit-learn

import os, json, math, numpy as np, torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import librosa
import gc

SR = 16000
MAX_DURATION = 5.0
MAX_LEN = int(MAX_DURATION * SR)
BATCH = 16
EPOCHS = 10
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load utterances
!wget -q -O /tmp/utt.jsonl.gz https://raw.githubusercontent.com/Das-rebel/chuck-audio-notebooks/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utt.jsonl.gz

utterances = [json.loads(l) for l in open('/tmp/utt.jsonl')]
print(f'Utterances: {len(utterances)}')

# Filter: duration > 0.05s and < 10s (relaxed from 5s to get more data)
utterances = [u for u in utterances if 0.05 < (u['end']-u['start']) < 10]
print(f'After filter: {len(utterances)}')

pos = sum(1 for u in utterances if u['label'] == 1)
print(f'Positive: {pos} ({100*pos/len(utterances):.2f}%)')

In [ ]:
# 3 comedian holdout (same as Kaggle)
HELD_OUT = {'BFIHCzw3itk', 'BAD4askmGgk', '1Nb3_os4RSA'}

held_out = [u for u in utterances if u['video_id'] in HELD_OUT]
in_dist = [u for u in utterances if u['video_id'] not in HELD_OUT]
print(f'In-dist: {len(in_dist)}, Held-out: {len(held_out)}')

# Shuffle in-dist
np.random.seed(SEED)
np.random.shuffle(in_dist)
n_train = int(len(in_dist) * 0.85)
train, val = in_dist[:n_train], in_dist[n_train:]
test = held_out

print(f'Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print(f'Train pos%: {100*sum(u["label"] for u in train)/len(train):.1f}%')
print(f'Val pos%: {100*sum(u["label"] for u in val)/len(val):.1f}%')
print(f'Test pos%: {100*sum(u["label"] for u in test)/len(test):.1f}%')

In [ ]:
# Find audio files
audio_files = {}
for root, dirs, files in os.walk('/content/gdrive/MyDrive/chuckle_audio_all'):
    for f in files:
        if f.endswith(('.mp3', '.wav')):
            vid = os.path.splitext(f)[0]
            if vid not in audio_files:
                audio_files[vid] = os.path.join(root, f)
print(f'Audio files: {len(audio_files)}')

In [ ]:
# Prosody extraction (same as Kaggle)
def extract_prosody(audio_path, start, end):
    """Extract 21-dim prosody features."""
    try:
        y, sr = librosa.load(audio_path, offset=start, duration=end-start, sr=SR)
        if len(y) < SR * 0.05:
            return np.zeros(21)
        
        feats = []
        
        # RMS (2)
        rms = librosa.feature.rms(y=y)[0]
        feats.extend([rms.mean(), rms.std()])
        
        # ZCR (2)
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        feats.extend([zcr.mean(), zcr.std()])
        
        # F0/pitch (2)
        try:
            pitches, mags = librosa.piptrack(y=y, sr=sr)
            pitch_vals = pitches[pitches > 0]
            if len(pitch_vals) > 0:
                feats.extend([pitch_vals.mean(), pitch_vals.std()])
            else:
                feats.extend([0, 0])
        except:
            feats.extend([0, 0])
        
        # MFCCs (13 dims x 2 = 26, but we only want 13 so just mean)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            feats.append(mfccs[i].mean())
        
        return np.array(feats[:21], dtype=np.float32)
    except:
        return np.zeros(21, dtype=np.float32)

In [ ]:
# Pre-extract prosody for train + val (test will be done during eval)
print('Extracting prosody for train + val...')
prosody_cache = {}

for split_name, data in [('train', train), ('val', val)]:
    for u in data:
        vid = u['video_id']
        if vid not in audio_files:
            continue
        key = f"{vid}_{u['start']:.2f}"
        if key in prosody_cache:
            continue
        prosody = extract_prosody(audio_files[vid], u['start'], u['end'])
        prosody_cache[key] = prosody
    print(f'  {split_name}: {len(prosody_cache)} prosody entries')

print(f'Total prosody entries: {len(prosody_cache)}')

In [ ]:
# StandardScaler on prosody (fit on train only)
train_prosody = np.array([prosody_cache.get(f"{u['video_id']}_{u['start']:.2f}", np.zeros(21)) for u in train])
prosody_scaler = StandardScaler()
prosody_scaler.fit(train_prosody)

def get_prosody(u):
    key = f"{u['video_id']}_{u['start']:.2f}"
    feats = prosody_cache.get(key, np.zeros(21))
    return prosody_scaler.transform([feats])[0]

In [ ]:
# Streaming audio loader (same as Kaggle)
def load_audio_segment(d, sr=SR, max_duration=MAX_DURATION):
    audio_path = f"{audio_files.get(d['video_id'], '')}"
    if not audio_path or not os.path.exists(audio_path):
        return np.zeros(int(max_duration * sr), dtype=np.float32)
    try:
        offset = max(0, d['start'] - 0.05)
        audio, _ = librosa.load(audio_path, sr=sr, mono=True, offset=offset, duration=max_duration)
        target_len = int(max_duration * sr)
        if len(audio) < target_len:
            audio = np.pad(audio, (0, target_len - len(audio)))
        return audio[:target_len]
    except:
        return np.zeros(int(max_duration * sr), dtype=np.float32)

In [ ]:
# Model (EXACT same as Kaggle)
print('Loading WavLM (FROZEN)...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm.eval()
for p in wavlm.parameters():
    p.requires_grad = False

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, hidden_states):
        weights = self.attn(hidden_states).squeeze(-1)
        weights = torch.softmax(weights, dim=-1)
        pooled = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return pooled, weights

class PhaseAWithProsody(nn.Module):
    def __init__(self, encoder, feat_extractor, prosody_dim=21, hidden=128):
        super().__init__()
        self.encoder = encoder
        self.feat_extractor = feat_extractor
        self.audio_pool = AttentionPooling(768)
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 32)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(800, hidden), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(hidden, 64), nn.GELU(), nn.Linear(64, 2)
        )
    def forward(self, waveforms, prosody_feats):
        # waveforms: (batch, 80000)
        wav_list = [w.cpu().numpy() for w in waveforms]
        inputs = self.feat_extractor(wav_list, sampling_rate=SR, return_tensors='pt', padding=False)
        inputs = {k: v.cpu() for k, v in inputs.items()}
        with torch.no_grad():
            hidden = self.encoder(**inputs).last_hidden_state.to(waveforms.device)
        audio_emb, _ = self.audio_pool(hidden)
        prosody_emb = self.prosody_proj(prosody_feats)
        fused = torch.cat([audio_emb, prosody_emb], dim=1)
        return self.classifier(fused)

model = PhaseAWithProsody(wavlm, feature_extractor).to(device)
model.encoder.to('cpu')  # P100 compatibility
torch.cuda.empty_cache()

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {n_total:,} total | {n_trainable:,} trainable')

In [ ]:
# Optimizer (same as Kaggle)
classifier_params = (
    list(model.audio_pool.parameters()) +
    list(model.prosody_proj.parameters()) +
    list(model.classifier.parameters())
)
optimizer = torch.optim.AdamW(classifier_params, lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.5]).to(device))
print('Optimizer ready')

In [ ]:
# Training functions (same as Kaggle)
def train_epoch(model, data, optimizer, criterion, device, epoch, batch_size=BATCH):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = math.ceil(len(data) / batch_size)
    np.random.seed(SEED + epoch)
    indices = np.random.permutation(len(data))
    
    for i in range(0, len(data), batch_size):
        batch_idx = indices[i:i+batch_size]
        batch = [data[j] for j in batch_idx]
        
        waveforms, prosody_feats, labels = [], [], []
        for d in batch:
            audio = load_audio_segment(d)
            waveforms.append(torch.FloatTensor(audio).unsqueeze(0))
            prosody_feats.append(get_prosody(d))
            labels.append(d['label'])
        
        waveforms = torch.stack(waveforms).squeeze(1).to(device)
        prosody_feats = torch.FloatTensor(np.array(prosody_feats)).to(device)
        labels = torch.LongTensor(labels).to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(waveforms, prosody_feats)
        loss = criterion(logits, labels)
        
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        del waveforms, prosody_feats, labels, logits, loss
        gc.collect()
    
    return total_loss / n_batches, f1_score(all_labels, all_preds, average='binary', zero_division=0)

def eval_epoch(model, data, criterion, device, batch_size=BATCH, extract_prosody_fn=None):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = math.ceil(len(data) / batch_size)
    
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        waveforms, prosody_feats, labels = [], [], []
        
        for d in batch:
            audio = load_audio_segment(d)
            waveforms.append(torch.FloatTensor(audio).unsqueeze(0))
            # For test, extract prosody on-the-fly
            prosody = extract_prosody(audio_files.get(d['video_id'], ''), d['start'], d['end']) if extract_prosody_fn else get_prosody(d)
            prosody_feats.append(prosody)
            labels.append(d['label'])
        
        waveforms = torch.stack(waveforms).squeeze(1).to(device)
        prosody_feats = torch.FloatTensor(np.array(prosody_feats)).to(device)
        labels = torch.LongTensor(labels).to(device)
        
        with torch.no_grad():
            logits = model(waveforms, prosody_feats)
            loss = criterion(logits, labels)
        
        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        del waveforms, prosody_feats, labels, logits, loss
        gc.collect()
    
    return total_loss / n_batches, f1_score(all_labels, all_preds, average='binary', zero_division=0), all_preds, all_labels

In [ ]:
# Train
print(f'Starting training...')
print(f'  Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print(f'  Batch: {BATCH}, Epochs: {EPOCHS}')

import time
t0 = time.time()
best_val_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    epoch_t0 = time.time()
    
    train_loss, train_f1 = train_epoch(model, train, optimizer, criterion, device, epoch)
    val_loss, val_f1, _, _ = eval_epoch(model, val, criterion, device)
    scheduler.step()
    
    improved = val_f1 > best_val_f1
    if improved:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    marker = ' BEST' if improved else ''
    print(f'Epoch {epoch+1:2d}/{EPOCHS} ({time.time()-epoch_t0:.0f}s){marker}')
    print(f'  Train loss={train_loss:.4f} f1={train_f1:.4f}')
    print(f'  Val   loss={val_loss:.4f} f1={val_f1:.4f}  (best={best_val_f1:.4f})')

print(f'\nTraining done! Best val F1: {best_val_f1:.4f}')
print(f'Total time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Evaluate on held-out test
model.load_state_dict(best_state)

print('Evaluating on held-out test...')
test_loss, test_f1, test_preds, test_labels = eval_epoch(model, test, criterion, device, extract_prosody_fn=True)

print(f'  Test F1={test_f1:.4f}')
print(f'  Precision={precision_score(test_labels, test_preds, zero_division=0):.4f}')
print(f'  Recall={recall_score(test_labels, test_preds, zero_division=0):.4f}')

In [ ]:
# Save
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_v2_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save(best_state, os.path.join(OUTPUT_DIR, 'model.pt'))

results = {
    'best_val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_precision': float(precision_score(test_labels, test_preds, zero_division=0)),
    'test_recall': float(recall_score(test_labels, test_preds, zero_division=0)),
    'n_train': len(train),
    'n_val': len(val),
    'n_test': len(test),
    'n_train_videos': len(set(u['video_id'] for u in train)),
    'n_val_videos': len(set(u['video_id'] for u in val)),
    'n_test_videos': len(set(u['video_id'] for u in test)),
}

with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')